### Build Races Dimension

1. Read silver `races` table
2. Read silver `circuits` table
3. Join the data from `races` with `circuits` using `circuit_id`
4. Select the required columns -  season, round, race_name, race_date (these are from races), circuit_name, locality, country (from circuits)
5. Write the transformed data to gold `dim_races` table

Below changes are required to implement Incremental Load Processing
1. Accept batch_id as a parameter to the notebook
2. Process data for only the batch_id being passed in (i.e., filter reading from silver using the batch_id)
3. Add created_timestamp, updated_timestamp to the gold table. 
4. Merge the processed data to the gold table
    - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
circuits_df = (
    spark.table(f"{catalog_name}.{silver_schema}.circuits")
        .filter(F.col("batch_id") == v_batch_id)
)

In [0]:

races_df = (
    spark.table(f"{catalog_name}.{silver_schema}.races")
        .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
dim_races_df = (
    races_df.alias("rc")
        .join(
            circuits_df.alias("crc")
            , F.col("rc.circuit_id") == F.col("crc.circuit_id")
            , "inner"
        )
        .selectExpr(
            "rc.season"
            , "rc.round"
            , "rc.race_name"
            , "rc.race_date"
            , "crc.circuit_name"
            , "crc.locality"
            , "crc.country"
        )
)
display(dim_races_df)

In [0]:
write_to_gold(
    input_df=dim_races_df
    , target_table=target_table
    , merge_condition="t.season = s.season AND t.round = s.round"
    , columns_to_update=[
        "race_name"
        , "race_date"
        , "circuit_name"
        , "locality"
        , "country"
    ]
)

In [0]:
display(spark.table(target_table))